In [14]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By

In [15]:
driver = webdriver.Chrome()

url = "https://www.evb.com/ko/faq/"
driver.get(url)

time.sleep(2)

In [16]:
faq_buttons = driver.find_elements(
    By.CSS_SELECTOR,
    "div.pp-accordion-toggle-icon > span.pp-accordion-toggle-icon-close.pp-icon > i"
)

print("FAQ 개수:", len(faq_buttons))

FAQ 개수: 13


In [17]:
data = []

for i in range(len(faq_buttons)):
    # stale 방지 (매번 재조회)
    faq_buttons = driver.find_elements(
        By.CSS_SELECTOR,
        "div.pp-accordion-toggle-icon > span.pp-accordion-toggle-icon-close.pp-icon > i"
    )
    
    button = faq_buttons[i]

    # 화면 이동
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
    time.sleep(1)

    # 아코디언 전체 컨테이너 찾기
    accordion_item = button.find_element(
        By.XPATH,
        "./ancestor::*[contains(@class, 'pp-accordion-item')][1]"
    )

    # 질문 추출
    question = ""
    try:
        question = accordion_item.find_element(
            By.XPATH,
            ".//*[contains(@class, 'pp-accordion-tab-title') or contains(@class, 'pp-accordion-title')][normalize-space()]"
        ).text.strip()
    except:
        pass

    # 클릭
    driver.execute_script("arguments[0].click();", button)
    time.sleep(2)

    # 답변 추출
    answer = ""
    try:
        answer = accordion_item.find_element(
            By.XPATH,
            ".//*[contains(@class, 'pp-accordion-tab-content') or contains(@class, 'pp-accordion-content')]"
        ).text.strip()
    except:
        pass

    # 유효 데이터만 저장
    if question and answer:
        data.append({
            "question": question,
            "answer": answer
        })

print("수집 완료:", len(data))

수집 완료: 13


In [18]:
for row in data[:5]:
    print("Q:", row["question"])
    print("A:", row["answer"])
    print("-" * 80)

Q: 전기 자동차 충전소의 경우, AC 충전과 DC 충전의 차이점은 무엇입니까?
A: AC 충전(레벨 2 충전이라고도 함)은 일반적으로 주택, 호텔 등 차량을 최소 몇 시간 이상 주차하는 장소에서 사용됩니다. DC 충전(레벨 3 충전이라고도 함)은 전기차를 훨씬 빠르게 충전할 수 있는 방법을 제공합니다. 이러한 충전기는 차량의 AC/DC 정류기에 의존하는 대신 내장된 전력 변환기를 사용하여 시간당 전기차에 공급할 수 있는 전력량(kW)으로 구분됩니다. DC 레벨 3 충전기는 한 시간 이내에 전기차를 완전히 충전할 수 있으며, 도로변 충전소, 차량 부대, 기타 전략적 위치에서 사용됩니다.더 보기
--------------------------------------------------------------------------------
Q: 전기자동차 충전소를 주문한 후 설치하는 데 얼마나 걸리나요?
A: 이는 여러 요인에 따라 달라집니다. 레벨 2 AC 충전기는 일반적으로 주문 접수 후 즉시 배송됩니다. 하지만 설치 과정은 현장 상황에 따라 달라집니다. 전기차 충전기 설치에는 계획 단계와 현장에 계획된 설치에 필요한 충분한 전력이 있는지 확인하는 검증 단계가 포함됩니다. 레벨 3 DC 충전기의 경우, 충전기 종류와 설치에 추가 인프라가 필요한지 여부에 따라 리드타임이 달라집니다.
--------------------------------------------------------------------------------
Q: 귀사의 EV 충전소를 이용해 전기 자동차를 완전히 충전하는 데 얼마나 걸리나요?
A: 이는 사용하는 충전기 유형, 차량이 수용할 수 있는 충전 속도, 차량 배터리의 현재 상태를 포함한 여러 요인에 따라 달라집니다.
--------------------------------------------------------------------------------
Q: 전기 자동차 충전소 도매 솔루션에 대한 구체적인 범위를 제공하시나요?
A:

In [20]:
df = pd.DataFrame(data)
df = df.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)

df.to_csv("evb_faq.csv", index=False, encoding="utf-8-sig")

print("CSV 저장 완료: evb_faq.csv")

CSV 저장 완료: evb_faq.csv
